# P0-3 · pandas: làm sạch và tổng hợp bảng dữ liệu

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import pandas as pd
s = pd.Series([1500, 400, 450], index=["Bàn A", "Ghế B", "Ghế C"], name="price")
s["Ghế B"]           # lấy theo nhãn: 400
s[s > 420]           # lọc bằng mặt nạ như numpy: Bàn A và Ghế C

400
Bàn A    1500
Ghế C     450


In [ ]:
import io
import pandas as pd

csv_text = """date,product,qty,price
2026-01-03,Bàn A,2,1500
2026-01-15, ghế b,1,400
2026-02-02,Bàn A,1,
2026-02-20,Bàn A,3,"1,500"
2026-02-20,Bàn A,3,"1,500"
2026-02-21,Ghế B,2,450
"""
raw = pd.read_csv(io.StringIO(csv_text))    # file thật: pd.read_csv("orders.csv"), pd.read_excel("orders.xlsx")
raw

         date product  qty  price
0  2026-01-03   Bàn A    2   1500
1  2026-01-15   ghế b    1    400
2  2026-02-02   Bàn A    1    NaN
3  2026-02-20   Bàn A    3  1,500
4  2026-02-20   Bàn A    3  1,500
5  2026-02-21   Ghế B    2    450


In [ ]:
raw.shape                        # (6, 4): 6 dòng, 4 cột
raw.dtypes                       # price là kiểu chữ vì có "1,500"; date cũng đang là chữ
raw.isna().sum()                 # số ô trống từng cột: price có 1
raw.duplicated().sum()           # 1 dòng trùng hoàn toàn với một dòng phía trên
raw["product"].value_counts()    # " ghế b" và "Ghế B" bị đếm thành hai sản phẩm

(6, 4)
date         str
product      str
qty        int64
price        str
date       0
product    0
qty        0
price      1
1
product
Bàn A     4
 ghế b    1
Ghế B     1


In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)              # bỏ dòng trùng, đánh lại index 0, 1, 2...
df["product"] = df["product"].str.strip().str.title()          # " ghế b" → "Ghế B"
df["price"] = pd.to_numeric(df["price"].str.replace(",", ""))  # "1,500" → 1500.0; ô trống vẫn là NaN
df["date"] = pd.to_datetime(df["date"])                        # chữ → kiểu ngày, dùng được .dt.month
df

        date product  qty   price
0 2026-01-03   Bàn A    2  1500.0
1 2026-01-15   Ghế B    1   400.0
2 2026-02-02   Bàn A    1     NaN
3 2026-02-20   Bàn A    3  1500.0
4 2026-02-21   Ghế B    2   450.0


In [ ]:
df.dtypes                         # date là datetime64, price là float64
df["product"].value_counts()      # còn đúng hai sản phẩm: Bàn A 3, Ghế B 2

date       datetime64[us]
product               str
qty                 int64
price             float64
product
Bàn A    3
Ghế B    2


In [ ]:
df.sort_values("qty", ascending=False)     # sắp theo qty giảm dần
df.nlargest(2, "qty")                      # 2 dòng có qty lớn nhất
df["product"].nunique()                    # 2: số giá trị khác nhau
df["product"].value_counts(normalize=True) # tỉ lệ thay vì số đếm: Bàn A 0.6, Ghế B 0.4

        date product  qty   price
3 2026-02-20   Bàn A    3  1500.0
0 2026-01-03   Bàn A    2  1500.0
4 2026-02-21   Ghế B    2   450.0
2 2026-02-02   Bàn A    1     NaN
1 2026-01-15   Ghế B    1   400.0
        date product  qty   price
3 2026-02-20   Bàn A    3  1500.0
0 2026-01-03   Bàn A    2  1500.0
2
product
Bàn A    0.6
Ghế B    0.4


In [ ]:
df[["product", "qty"]]                     # nhiều cột → DataFrame
df.loc[df["qty"] >= 2, ["product", "qty"]] # loc: lọc theo điều kiện và tên cột
df.iloc[0:2, 1:3]                          # iloc: theo vị trí, giống slicing của numpy

  product  qty
0   Bàn A    2
1   Ghế B    1
2   Bàn A    1
3   Bàn A    3
4   Ghế B    2
  product  qty
0   Bàn A    2
3   Bàn A    3
4   Ghế B    2
  product  qty
0   Bàn A    2
1   Ghế B    1


In [ ]:
len(df[df["qty"] >= 2 & (df["price"] > 1000)])      # sai: ra 5 dòng
len(df[(df["qty"] >= 2) & (df["price"] > 1000)])    # đúng: ra 2 dòng

5
2


In [ ]:
df["price"] = df["price"].fillna(df.groupby("product")["price"].transform("median"))
df["revenue"] = df["qty"] * df["price"]     # cột mới, tính trên cả cột (vector hóa)
df

        date product  qty   price  revenue
0 2026-01-03   Bàn A    2  1500.0   3000.0
1 2026-01-15   Ghế B    1   400.0    400.0
2 2026-02-02   Bàn A    1  1500.0   1500.0
3 2026-02-20   Bàn A    3  1500.0   4500.0
4 2026-02-21   Ghế B    2   450.0    900.0


In [ ]:
df.groupby("product")["revenue"].sum()
# Bàn A    9000.0      ← 3000 + 1500 + 4500
# Ghế B    1300.0      ← 400 + 900

product
Bàn A    9000.0
Ghế B    1300.0


In [ ]:
df.groupby("product").agg(
    so_don=("qty", "size"),       # số dòng mỗi nhóm
    tong_sl=("qty", "sum"),
    doanh_thu=("revenue", "sum"),
)

         so_don  tong_sl  doanh_thu
product                            
Bàn A         3        6     9000.0
Ghế B         2        3     1300.0


In [ ]:
df.groupby(df["date"].dt.to_period("M"))["revenue"].sum()
# 2026-01    3400.0
# 2026-02    6900.0

date
2026-01    3400.0
2026-02    6900.0
Freq: M


In [ ]:
df.pivot_table(index=df["date"].dt.month, columns="product",
               values="revenue", aggfunc="sum", fill_value=0)

product   Bàn A  Ghế B
date                  
1        3000.0  400.0
2        6000.0  900.0


In [ ]:
products = pd.DataFrame({"product": ["Bàn A", "Ghế B"], "category": ["Bàn", "Ghế"]})
full = df.merge(products, on="product", how="left")   # thêm cột category vào mỗi đơn
full[["date", "product", "category"]]

        date product category
0 2026-01-03   Bàn A      Bàn
1 2026-01-15   Ghế B      Ghế
2 2026-02-02   Bàn A      Bàn
3 2026-02-20   Bàn A      Bàn
4 2026-02-21   Ghế B      Ghế


In [ ]:
jan = df[df["date"].dt.month == 1]
feb = df[df["date"].dt.month == 2]
pd.concat([jan, feb], ignore_index=True).shape    # (5, 5): lại đủ 5 đơn

(5, 5)


In [ ]:
dup = pd.DataFrame({"product": ["Bàn A", "Bàn A"], "color": ["trắng", "đen"]})
len(df), len(df.merge(dup, on="product", how="left"))   # (5, 8)

(5, 8)


In [ ]:
df["month"] = df["date"].dt.month                     # [1, 1, 2, 2, 2]
df["weekday"] = df["date"].dt.dayofweek               # 0 = thứ Hai, ..., 6 = Chủ nhật
df["is_table"] = df["product"].str.contains("Bàn")    # True/False
df["size"] = pd.cut(df["revenue"], bins=[0, 1000, 3000, 10_000], labels=["nhỏ", "vừa", "lớn"])
df[["product", "revenue", "month", "weekday", "is_table", "size"]]

  product  revenue  month  weekday  is_table size
0   Bàn A   3000.0      1        5      True  vừa
1   Ghế B    400.0      1        3     False  nhỏ
2   Bàn A   1500.0      2        0      True  vừa
3   Bàn A   4500.0      2        4      True  lớn
4   Ghế B    900.0      2        5     False  nhỏ


In [ ]:
import numpy as np, time
big = pd.DataFrame({"qty": np.random.default_rng(0).integers(1, 5, 200_000), "price": 450.0})
t = time.time(); r1 = big.apply(lambda row: row["qty"] * row["price"], axis=1); t1 = time.time() - t
t = time.time(); r2 = big["qty"] * big["price"]; t2 = time.time() - t
print(f"apply: {t1:.2f} s, vector hóa: {t2 * 1000:.1f} ms, cùng kết quả: {r1.equals(r2)}")

apply: 0.72 s, vector hóa: 1.9 ms, cùng kết quả: True


In [ ]:
df.to_csv("orders_clean.csv", index=False)       # index=False: không ghi cột số thứ tự
pd.read_csv("orders_clean.csv", parse_dates=["date"]).shape   # đọc lại đủ (5, 9)

(5, 9)


## Bài tập 3: Làm sạch tên sản phẩm {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
names = pd.Series([" Bàn A", "bàn a", "BÀN A ", "Ghế B", "ghế  b", None])

In [ ]:
import pandas as pd
names = pd.Series([" Bàn A", "bàn a", "BÀN A ", "Ghế B", "ghế  b", None])
clean = names.str.strip().str.split().str.join(" ").str.title()
print(clean.value_counts())
print("thiếu tên:", clean.isna().sum())

Bàn A    3
Ghế B    2
Name: count, dtype: int64
thiếu tên: 1


## Bài tập 4: Doanh thu theo tháng và tỉ lệ hoàn {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "date": pd.to_datetime(["2026-03-02", "2026-03-15", "2026-03-20", "2026-04-01", "2026-04-09", "2026-04-28"]),
    "product": ["Bàn A", "Gương", "Gương", "Bàn A", "Gương", "Bàn A"],
    "qty": [1, 2, 1, 2, 1, 1],
    "price": [1500, 300, 300, 1500, 300, 1500],
    "returned": [False, True, False, False, False, True],
})

In [ ]:
orders["revenue"] = orders["qty"] * orders["price"]
sold = orders[~orders["returned"]]
print(sold.groupby(sold["date"].dt.to_period("M"))["revenue"].sum())
print(orders.groupby("product")["returned"].mean().sort_values(ascending=False))

date
2026-03    1800
2026-04    3300
Freq: M, Name: revenue, dtype: int64
product
Bàn A    0.333333
Gương    0.333333
Name: returned, dtype: float64


## Bài tập 5: Khám phá file khách hàng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import io
import pandas as pd
csv_text = """customer_id,name,city,age,signup
C01,Lan,Hà Nội,25,2025-03-02
C02,Minh,TP.HCM,31,2025-04-11
C03,Hoa,Hà Nội,,2025-04-20
C04,Tuấn,Đà Nẵng,42,2025-05-01
C02,Minh,TP.HCM,31,2025-04-11
C05,An,Hà Nội,19,2025-06-15
C06,Bình,TP.HCM,,2025-06-30
"""
kh = pd.read_csv(io.StringIO(csv_text))
# TODO: xem kh.shape, kh.dtypes, kh.isna().sum(), kh.duplicated().sum(), kh["city"].value_counts()

In [ ]:
print(kh.shape)
print(kh.dtypes)
print(kh.isna().sum())
print("trùng:", kh.duplicated().sum())
kh_sach = kh.drop_duplicates()
print(kh_sach["city"].value_counts())
print(kh_sach["age"].describe())

(7, 5)
customer_id        str
name               str
city               str
age            float64
signup             str
dtype: object
customer_id    0
name           0
city           0
age            2
signup         0
dtype: int64
trùng: 1
city
Hà Nội     3
TP.HCM     2
Đà Nẵng    1
Name: count, dtype: int64
count     4.000000
mean     29.250000
std       9.810708
min      19.000000
25%      23.500000
50%      28.000000
75%      33.750000
max      42.000000
Name: age, dtype: float64


## Bài tập 6: Lọc khách cần chăm sóc lại {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
kh = pd.DataFrame({
    "name": ["Lan", "Minh", "Hoa", "Tuấn", "An", "Bình", "Chi"],
    "city": ["Hà Nội", "TP.HCM", "Hà Nội", "Đà Nẵng", "Hà Nội", "TP.HCM", "Huế"],
    "total_spent": [3.5, 1.2, 5.0, 4.1, 2.4, 2.8, 6.0],        # triệu đồng
    "days_since_last": [90, 120, 30, 200, 75, 61, 300],
})
kh["can_goi"] = False
# TODO: dieu_kien, ds_goi

In [ ]:
dieu_kien = (kh["city"].isin(["Hà Nội", "TP.HCM"])) & (kh["total_spent"] > 2) & (kh["days_since_last"] > 60)
ds_goi = kh.loc[dieu_kien, ["name", "city", "total_spent"]].sort_values("total_spent", ascending=False)
kh.loc[dieu_kien, "can_goi"] = True
print(ds_goi)

   name    city  total_spent
0   Lan  Hà Nội          3.5
5  Bình  TP.HCM          2.8
4    An  Hà Nội          2.4


## Bài tập 7: Điền ô trống có lý do {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import pandas as pd
can_ho = pd.DataFrame({
    "bedrooms": [1, 1, 2, 2, 2, 3, 3, 1, 2, 3],
    "area": [35, 40, np.nan, 68, 72, 95, np.nan, np.nan, 65, 102],
    "price": [1.6, 1.9, 2.9, 3.1, 3.3, 4.6, 4.4, 1.7, 3.0, 5.1],     # tỷ đồng
})
# TODO: tb_truoc, area_missing, điền ô trống, tb_sau

In [ ]:
tb_truoc = can_ho["area"].mean()
can_ho["area_missing"] = can_ho["area"].isna()
can_ho["area"] = can_ho["area"].fillna(can_ho.groupby("bedrooms")["area"].transform("median"))
tb_sau = can_ho["area"].mean()
print(can_ho)
print(f"trung bình trước: {tb_truoc:.2f}, sau: {tb_sau:.2f}")

   bedrooms   area  price  area_missing
0         1   35.0    1.6         False
1         1   40.0    1.9         False
2         2   68.0    2.9          True
3         2   68.0    3.1         False
4         2   72.0    3.3         False
5         3   95.0    4.6         False
6         3   98.5    4.4          True
7         1   37.5    1.7          True
8         2   65.0    3.0         False
9         3  102.0    5.1         False
trung bình trước: 68.14, sau: 68.10


## Bài tập 8: Báo cáo chéo tháng × danh mục {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
don = pd.DataFrame({
    "date": pd.to_datetime(["2026-01-05", "2026-01-18", "2026-01-25", "2026-02-03", "2026-02-14", "2026-02-20", "2026-03-01", "2026-03-09"]),
    "category": ["Bàn", "Gương", "Bàn", "Ghế", "Bàn", "Gương", "Ghế", "Bàn"],
    "revenue": [3000, 450, 1500, 500, 4500, 900, 750, 3000],
})
# TODO: bang (pivot), ti_trong

In [ ]:
bang = pd.pivot_table(don, index=don["date"].dt.month, columns="category", values="revenue", aggfunc="sum", fill_value=0)
ti_trong = bang.div(bang.sum(axis=1), axis=0) * 100
bang["Tổng"] = bang.sum(axis=1)
print(bang)
print(ti_trong.round(1))

category   Bàn  Ghế  Gương  Tổng
date                            
1         4500    0    450  4950
2         4500  500    900  5900
3         3000  750      0  3750
category   Bàn   Ghế  Gương
date                       
1         90.9   0.0    9.1
2         76.3   8.5   15.3
3         80.0  20.0    0.0


## Bài tập 9: Ghép đơn hàng với khách hàng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
thang1 = pd.DataFrame({"order_id": [1, 2, 3], "customer_id": ["C01", "C02", "C01"], "revenue": [3000, 450, 1500]})
thang2 = pd.DataFrame({"order_id": [4, 5, 6], "customer_id": ["C03", "C09", "C02"], "revenue": [500, 4500, 900]})
khach = pd.DataFrame({"customer_id": ["C01", "C02", "C03"], "name": ["Lan", "Minh", "Hoa"], "city": ["Hà Nội", "TP.HCM", "Hà Nội"]})
# TODO: don (gộp 2 tháng), full (ghép khách), loi (đơn không khớp)

In [ ]:
don = pd.concat([thang1, thang2], ignore_index=True)
full = don.merge(khach, on="customer_id", how="left", validate="many_to_one", indicator=True)
loi = full[full["_merge"] == "left_only"]
print(full)
print("đơn lỗi:", loi[["order_id", "customer_id"]].to_dict("records"))

   order_id customer_id  revenue  name    city     _merge
0         1         C01     3000   Lan  Hà Nội       both
1         2         C02      450  Minh  TP.HCM       both
2         3         C01     1500   Lan  Hà Nội       both
3         4         C03      500   Hoa  Hà Nội       both
4         5         C09     4500   NaN     NaN  left_only
5         6         C02      900  Minh  TP.HCM       both
đơn lỗi: [{'order_id': 5, 'customer_id': 'C09'}]


## Bài tập 10: Đặc trưng thời gian cho dự báo đơn hàng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
don = pd.DataFrame({
    "time": pd.to_datetime(["2026-03-02 09:15", "2026-03-03 20:40", "2026-03-07 10:05", "2026-03-07 21:30",
                            "2026-03-08 14:20", "2026-03-09 23:50", "2026-03-10 19:10", "2026-03-14 20:05"]),
    "revenue": [450, 1500, 3000, 900, 1500, 300, 450, 3000],
})
# TODO: weekday, is_weekend, khung_gio, tb_theo_ngay, khung_nhieu_nhat

In [ ]:
don["weekday"] = don["time"].dt.dayofweek
don["is_weekend"] = don["weekday"] >= 5
don["khung_gio"] = pd.cut(don["time"].dt.hour, bins=[0, 6, 12, 18, 24], right=False, labels=["đêm", "sáng", "chiều", "tối"])
tb_theo_ngay = don.groupby("is_weekend")["revenue"].mean()
khung_nhieu_nhat = don["khung_gio"].value_counts().idxmax()
print(don)
print(tb_theo_ngay, "khung giờ nhiều đơn nhất:", khung_nhieu_nhat)

                 time  revenue  weekday  is_weekend khung_gio
0 2026-03-02 09:15:00      450        0       False      sáng
1 2026-03-03 20:40:00     1500        1       False       tối
2 2026-03-07 10:05:00     3000        5        True      sáng
3 2026-03-07 21:30:00      900        5        True       tối
4 2026-03-08 14:20:00     1500        6        True     chiều
5 2026-03-09 23:50:00      300        0       False       tối
6 2026-03-10 19:10:00      450        1       False       tối
7 2026-03-14 20:05:00     3000        5        True       tối
is_weekend
False     675.0
True     2100.0
Name: revenue, dtype: float64 khung giờ nhiều đơn nhất: tối
